# 第 81 课｜推理服务灾难恢复、持久快照与双实例一致性

从上一课的单机发布日志走向**两个逻辑推理副本**。这次做的是可以重放、可以故障注入、可以检查资源是否释放的恢复演练。

**目标路线：** 真实增量 KV → 逻辑会话 WAL → 快照 → 同步/异步确认 → 副本恢复 → epoch 围栏 → RPO/RTO 检查 → 故障报告。

**严格边界：** 双实例在同一个 Python 进程内运行，各有独立模型对象、实际 CPU KV tensor 和独立本地文件目录；不是远程网络服务，没有分布式共识/强一致租约/真实分布式原子切换。`fsync` 模拟每个本地实例的持久 ACK 条件；它无法证明两台物理机器的失效域独立。

所有 RPO/RTO **tick 都是本课人为定义的离散成本/事件时间**，不是秒、真实请求延迟或可用于生产 SLA 的成绩。模型随机初始化，不把生成文本当作有用回答。

## 0. 运行方式

在现有 PyTorch 环境的 Jupyter 内核中，**重启并运行全部**。本讲义包含全部类与函数，独立于配套脚本和前面课程文件，不下载任何模型。

完整演练默认执行10个场景，在工作目录下创建 `lesson_81_outputs/run_时间戳`，**不覆盖已有文件**。可修改 `RUN_FULL_EXPERIMENT=False` 先做数学与人工练习。

首次请重点看4条约束：

- ACK 只代表你明确规定的持久条件；`sync` 与 `async` 不同。
- 一个副本的 WAL/快照逻辑状态，不等于已经拥有能继续生成的 KV 张量。
- 接管必须检查**模型权重、模板、日志前缀与 epoch**；版本不符拒绝。
- 同步确认后的丢失量和实际 RPO 上限属于条件性声明；本实验是进程/本地文件模拟。

这里直接写入的逻辑 token ID 可能重建原文，**没有加密、真实鉴权或数据脱敏保证**；不要用生产请求数据运行。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第81课：推理服务灾难恢复、检查点与双实例一致性。

完整离线、CPU PyTorch：真实自回归Transformer、Q/V LoRA与物理KV张量。
演示独立副本、同步/异步确认、逻辑会话WAL、快照恢复与epoch围栏。
仅为单进程模拟的双实例；不提供真实集群共识、远端存储、网络或IAM。
所有RTO/RPO单位是显式的教学逻辑tick；不宣称GPU毫秒性能。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import tempfile
import time
from collections import Counter
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit('需要已有PyTorch环境；不会自动安装或下载。') from exc

VERSION = 'lesson81-dr-replication-v1'
SOURCES = {
    'aws_objectives':'https://docs.aws.amazon.com/wellarchitected/latest/reliability-pillar/disaster-recovery-dr-objectives.html',
    'google_recovery':'https://docs.cloud.google.com/architecture/framework/reliability/perform-testing-for-recovery-from-data-loss',
    'google_dr':'https://docs.cloud.google.com/architecture/disaster-recovery',
    'aws_dr':'https://docs.aws.amazon.com/wellarchitected/latest/framework/rel_planning_for_recovery_objective_defined_recovery.html',
    'torch_random':'https://docs.pytorch.org/docs/stable/notes/randomness.html',
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""


def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")


def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
SEED = 8101
DEVICE = 'cpu'
RUN_FULL_EXPERIMENT = True
OUTPUT_ROOT = Path('lesson_81_outputs')
seed_all(SEED)
print('PyTorch',torch.__version__,'CUDA可用:',torch.cuda.is_available())
print('本课默认只使用CPU；不测试RTX 3060耗时或显存。')

PyTorch 2.10.0+cpu CUDA可用: False
本课默认只使用CPU；不测试RTX 3060耗时或显存。


## 1. 真实模型和分词器的身份是什么？

复用前几课的教学字节级分词器、随机自回归 Transformer 和 Q/V LoRA，但**完整代码复制到本课**，不会导入以前的 `.py`。

分词器有独立的角色和终止 token；普通文本按 UTF-8 字节编码并支持固定合并规则。它不是 Hugging Face 的真实 FastTokenizer 或通用 Chat Template。模型隐藏维度16，2个注意力头、1层 Transformer，两个版本使用同一基座但 adapter 不同。

两个逻辑副本各自持有一份完整的模型对象；身份由分词协议、结构和权重内容共同确定。`stable` 和 `canary` 的输出不应该混为同一KV缓存。

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}


@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')


class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)


class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))


class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)


def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy


def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]


def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}


def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))


def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base


def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

## 2. 为什么不直接把整个 KV 缓存保存进快照？

`PagedKVPool` 的实际存储是 CPU tensor，形状为：

```text
[物理块, Transformer层, K/V, 块内位置, 注意力头, 单头维度]
```

增量 forward 每新增一个 token，写入当前层的 K/V，然后从已写入的块表读取历史 K/V。完整前向与增量输出必须数值一致。

**本课刻意不保存物理块 ID 或 KV tensor。** 它们与池大小、模型权重、位置、共享引用、设备状态等绑定，直接跨实例恢复风险很大。只持久化请求版本、提示 token ID、已提交生成 token ID 和停止状态；换副本后从这些逻辑数据**重算** KV。

这使恢复计算成本非零，但显式避免把一个进程的页指针当成另一个进程可用的指针。

In [4]:
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u


class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [5]:
def build_servable_models(seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    seed_all(seed)
    tok=ByteMergeChatTokenizer()
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32,max_length=96)
    base=TinyChatLM(cfg).eval()
    stable=inject_lora(base,rank=4,alpha=8.0)
    canary=inject_lora(base,rank=4,alpha=8.0)
    gen=torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for name,p in adapters(canary):
            if name.endswith('.B'):
                p.copy_(torch.randn(p.shape,generator=gen)*.1)
    models={'stable':stable.eval().requires_grad_(False),'canary':canary.eval().requires_grad_(False)}
    require(base_hash(models['stable'])==base_hash(models['canary']),'不是同一基座')
    require(state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()),'两个版本实际权重相同')
    return tok,models

In [6]:
tok, models = build_servable_models(SEED)
model_id = artifact_identity(tok, models) if 'artifact_identity' in globals() else None
print('教学分词器词表大小:',tok.vocab_size)
print('模型版本:',list(models))
print('每个版本的参数数量:',sum(p.numel() for p in models['stable'].parameters()))
print('stable/canary原始权重不同:',state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()))
print('同一基座:',base_hash(models['stable'])==base_hash(models['canary']))

教学分词器词表大小: 281
模型版本: ['stable', 'canary']
每个版本的参数数量: 12976
stable/canary原始权重不同: True
同一基座: True


## 3. WAL：先记录可重放事实，再向客户端承诺

WAL（Write-Ahead Log）每行只有两类教学事件：

```text
create：租户键、幂等键、版本、提示 IDs、输出预算
token ：请求 ID、token 序号、真实输出 token ID、终止原因
```

每条记录有从1开始的连续 `seq`、相邻事件推进1的**逻辑 tick**、前驱哈希与自身哈希。写盘后 `flush`＋`fsync`，重启时要求每行JSON完整、以换行结束、哈希链连续且事件状态转换有效。

**裸 SHA-256 哈希链不是数字签名**；有能力改写整个日志和所有hash的人仍可能伪造历史。因此快照同时要求与可验证 WAL 前缀的完整逻辑状态一致。本课没有外部签名、不可篡改存储或真实数据权限控制。

In [7]:
ZERO_HASH='0'*64

@dataclass
class LogicalRequest:
    rid: str
    tenant: str
    key: str
    version: str
    prompt_ids: list[int]
    max_new_tokens: int
    generated: list[int]=field(default_factory=list)
    terminal: str|None=None

    def identity(self):
        return {'tenant':self.tenant,'key':self.key,'version':self.version,
                'prompt_ids':self.prompt_ids,'max_new_tokens':self.max_new_tokens}


class Ledger:
    """WAL顺序应用的逻辑状态，不包含任何模型权重或物理KV。"""
    def __init__(self,codec:ByteMergeChatTokenizer,model_identity:dict):
        self.codec=codec
        self.model_identity=copy.deepcopy(model_identity)
        self.sessions:dict[str,LogicalRequest]={}
        self.seq=0;self.last_hash=ZERO_HASH;self.tick=0

    def to_json(self):
        return {'seq':self.seq,'last_hash':self.last_hash,'tick':self.tick,
                'sessions':{key:asdict(state) for key,state in sorted(self.sessions.items())}}

    def apply(self,record:dict):
        require(type(record) is dict and set(record)=={'seq','tick','kind','payload','prev','hash'},
                'WAL字段集合错误')
        require(type(record['seq']) is int and record['seq']==self.seq+1,'WAL序号必须连续')
        require(type(record['tick']) is int and record['tick']==self.tick+1,
                '本课tick是从1开始的每条持久事件序号')
        require(record['prev']==self.last_hash,'WAL前驱hash错误或从错误分叉接入')
        require(record['hash']==digest({k:v for k,v in record.items() if k!='hash'}),
                'WAL内容hash错误')
        kind=record['kind'];p=record['payload']
        require(type(p) is dict,'payload必须是dict')
        if kind=='create':
            require(set(p)=={'rid','tenant','key','version','prompt_ids','max_new_tokens'},
                    'create字段集合错误')
            rid=make_rid(p['tenant'],p['key'])
            require(rid==p['rid'] and rid not in self.sessions,'重复idempotency key/create冲突')
            require(p['version'] in self.model_identity['weights'],'WAL引用未知权重版本')
            require(type(p['prompt_ids']) is list and len(p['prompt_ids'])>0 and
                    len(p['prompt_ids'])<self.model_identity['max_length'], '提示token非法/超上下文')
            require(all(type(t) is int and 0<=t<self.codec.vocab_size for t in p['prompt_ids']),
                    'WAL提示token超词表')
            require(type(p['max_new_tokens']) is int and 1<=p['max_new_tokens']<=4 and
                    len(p['prompt_ids'])+p['max_new_tokens']<=self.model_identity['max_length'],
                    'WAL完成长度非法')
            self.sessions[rid]=LogicalRequest(**p)
        elif kind=='token':
            require(set(p)=={'rid','index','token','terminal'},'token字段集合错误')
            rid=p['rid'];require(rid in self.sessions,'追加token的请求不存在')
            session=self.sessions[rid]
            require(session.terminal is None,'终态后不可再追加token')
            require(type(p['index']) is int and p['index']==len(session.generated),
                    'token序号不连续')
            tok=p['token'];require(type(tok) is int and 0<=tok<self.codec.vocab_size,
                                  'token ID越界')
            require(p['terminal'] in {None,'eos','length'},'非法停止原因')
            next_length=len(session.generated)+1
            expected='eos' if tok==self.codec.eos_token_id else (
                     'length' if next_length==session.max_new_tokens else None)
            require(p['terminal']==expected,'EOS、长度终止不一致')
            session.generated.append(tok)
            session.terminal=p['terminal']
        else:
            raise LessonError('WAL事件类型非法')
        self.seq=record['seq'];self.tick=record['tick'];self.last_hash=record['hash']

    @classmethod
    def from_json(cls,codec,model_identity,data):
        require(type(data) is dict and set(data)=={'seq','last_hash','tick','sessions'},
                '快照状态字段错误')
        out=cls(codec,model_identity)
        require(type(data['seq']) is int and data['seq']>=0 and
                data['tick']==data['seq'] and type(data['last_hash']) is str and
                len(data['last_hash'])==64,'快照head错误')
        out.seq=data['seq'];out.tick=data['tick'];out.last_hash=data['last_hash']
        require(type(data['sessions']) is dict,'快照会话表必须为dict')
        for rid,s in data['sessions'].items():
            require(type(s) is dict and set(s)==set(LogicalRequest.__dataclass_fields__),
                    '快照会话字段非法')
            item=LogicalRequest(**s)
            require(item.rid==rid==make_rid(item.tenant,item.key),'快照id不一致')
            require(item.version in model_identity['weights'],'快照版本已失效')
            require(item.terminal in {None,'eos','length'},'快照停止状态非法')
            require(all(type(i) is int and 0<=i<codec.vocab_size for i in
                        item.prompt_ids+item.generated),'快照token越界')
            require(1<=item.max_new_tokens<=4 and
                    len(item.generated)<=item.max_new_tokens,'快照生成长度非法')
            if item.terminal=='eos':
                require(bool(item.generated) and item.generated[-1]==codec.eos_token_id,
                        '快照EOS状态不一致')
            if item.terminal=='length':
                require(len(item.generated)==item.max_new_tokens and
                        item.generated[-1]!=codec.eos_token_id,'快照长度终止不一致')
            if item.terminal is None:
                require(codec.eos_token_id not in item.generated and
                        len(item.generated)<item.max_new_tokens,'活动快照包含终态回答')
            out.sessions[rid]=item
        return out


def make_rid(tenant:str,key:str) -> str:
    for value,label in ((tenant,'租户'),(key,'幂等键')):
        require(type(value) is str and re.fullmatch('[A-Za-z0-9_-]{1,18}',value) is not None,
                label+'只能含1..18个ASCII字母数字_-')
    return tenant+'__'+key


def atomic_json(path:Path,payload:dict) -> None:
    """单机持久文件替换＋fsync；没有提供跨机器线性一致性。"""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    require(not tmp.exists(),'存在未清理的临时文件，拒绝覆盖')
    try:
        with tmp.open('x',encoding='utf-8') as f:
            f.write(json.dumps(payload,ensure_ascii=False,sort_keys=True,allow_nan=False)+'\n')
            f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        if hasattr(os,'O_DIRECTORY'):
            fd=os.open(str(path.parent),os.O_RDONLY|os.O_DIRECTORY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if tmp.exists():tmp.unlink()


def make_record(ledger:Ledger,kind:str,payload:dict) -> dict:
    row={'seq':ledger.seq+1,'tick':ledger.tick+1,'kind':kind,
         'payload':copy.deepcopy(payload),'prev':ledger.last_hash}
    row['hash']=digest(row)
    trial=copy.deepcopy(ledger)
    trial.apply(row)
    return row


class DurableJournal:
    """严格顺序WAL：每行必须换行、hash chain，追加时flush+fsync。

    这是本地模拟“两个独立实例的磁盘”，不是分布式事务，不跨网复制fsync保证。
    故障后的未完成行默认拒绝；不静默截断或忽略。
    """
    def __init__(self,folder:Path,codec,model_identity,load_existing=False):
        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
        self.path=self.folder/'events.jsonl'
        self.snapshot_path=self.folder/'snapshot.json'
        self.codec=codec;self.model_identity=copy.deepcopy(model_identity)
        if not load_existing:
            require(not self.path.exists() and not self.snapshot_path.exists(),
                    '实例目录已有数据，禁止当作新副本覆盖')
            self.path.write_bytes(b'')
        else:
            require(self.path.is_file(),'恢复时WAL不存在')
        self.records=[];self.ledger=Ledger(codec,model_identity)
        if load_existing:self.recover()

    def read_records(self) -> list[dict]:
        raw=self.path.read_bytes()
        require(not raw or raw.endswith(b'\n'),'WAL末尾存在截断行，必须人工修复/显式恢复流程')
        rows=[];temp=Ledger(self.codec,self.model_identity)
        for n,line in enumerate(raw.splitlines(),1):
            require(line!=b'','WAL存在空行')
            try:
                row=json.loads(line.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError(f'WAL第{n}行不是有效JSON/UTF8') from exc
            temp.apply(row)
            rows.append(row)
        return rows

    def recover(self) -> dict:
        rows=self.read_records()  # 对全WAL做语法/hash链验证，快照不绕过完整性审计。
        all_state=Ledger(self.codec,self.model_identity)
        for row in rows:all_state.apply(row)
        snapshot_used=False;tail=rows
        if self.snapshot_path.exists():
            try:wrapped=json.loads(self.snapshot_path.read_text(encoding='utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError('快照文件无法解码') from exc
            require(type(wrapped) is dict and set(wrapped)=={'payload','sha256'},'快照封装非法')
            data=wrapped['payload']
            require(wrapped['sha256']==digest(data),'快照内容摘要不匹配')
            require(type(data) is dict and set(data)=={'version','model_identity','state'},
                    '快照字段错误')
            require(data['version']==VERSION and data['model_identity']==self.model_identity,
                    '快照模型/分词身份不匹配')
            restored=Ledger.from_json(self.codec,self.model_identity,data['state'])
            require(restored.seq<=len(rows),'快照seq超出WAL实际长度')
            prefix=Ledger(self.codec,self.model_identity)
            for row in rows[:restored.seq]:prefix.apply(row)
            require(prefix.to_json()==restored.to_json(),
                    '快照与WAL前缀的内容不一致：不允许信任重算hash后的假快照')
            tail=rows[restored.seq:]
            for row in tail:restored.apply(row)
            require(restored.to_json()==all_state.to_json(),'快照加尾WAL不等于全量恢复')
            self.ledger=restored;snapshot_used=True
        else:self.ledger=all_state
        self.records=rows
        return {'wal_records_verified':len(rows),'snapshot_used':snapshot_used,
                'tail_events_semantically_applied':len(tail),'head_hash':self.ledger.last_hash}

    def append_row(self,row:dict) -> None:
        # 在触盘前先检查下一状态，避免错误事件落盘后再被拒绝。
        probe=copy.deepcopy(self.ledger);probe.apply(row)
        with self.path.open('ab') as f:
            payload=(canonical(row)+'\n').encode('utf-8')
            f.write(payload);f.flush();os.fsync(f.fileno())
        self.ledger=probe
        self.records.append(copy.deepcopy(row))

    def checkpoint(self) -> dict:
        data={'version':VERSION,'model_identity':copy.deepcopy(self.model_identity),
              'state':self.ledger.to_json()}
        atomic_json(self.snapshot_path,{'payload':data,'sha256':digest(data)})
        return {'seq':self.ledger.seq,'last_hash':self.ledger.last_hash,
                'size_bytes':self.snapshot_path.stat().st_size,
                'contents':'logical token IDs and request states; NOT raw KV tensors'}

In [8]:
# 独立WAL数学练习：无需模型生成，可逐行理解可重放事实。
with tempfile.TemporaryDirectory() as temporary:
    sample_id = {'tokenizer':digest(tok.contract()),
                 'weights':{'stable':state_hash(models['stable'].state_dict())},
                 'max_length':models['stable'].cfg.max_length}
    ledger = Ledger(tok,sample_id)
    prompt = tok.apply_chat_template([{'role':'user','content':'hello'}],
                                     tokenize=True,add_generation_prompt=True)
    payload = {'rid':'class__demo','tenant':'class','key':'demo','version':'stable',
               'prompt_ids':prompt,'max_new_tokens':3}
    row = make_record(ledger,'create',payload)
    ledger.apply(row)
    journal = DurableJournal(Path(temporary),tok,sample_id)
    journal.append_row(row)
    checkpoint = journal.checkpoint()
    replayed = DurableJournal(Path(temporary),tok,sample_id,load_existing=True)
    print('WAL第1行:', {k:row[k] for k in ('seq','tick','kind','prev','hash')})
    print('快照:',checkpoint)
    print('重载后逻辑状态完全相等:',ledger.to_json()==replayed.ledger.to_json())

WAL第1行: {'seq': 1, 'tick': 1, 'kind': 'create', 'prev': '0000000000000000000000000000000000000000000000000000000000000000', 'hash': '91d2f39bc04112313b35bc73051c936b0930274b2c32bfdf2e57a64788436447'}
快照: {'seq': 1, 'last_hash': '91d2f39bc04112313b35bc73051c936b0930274b2c32bfdf2e57a64788436447', 'size_bytes': 659, 'contents': 'logical token IDs and request states; NOT raw KV tensors'}
重载后逻辑状态完全相等: True


## 4. Snapshot 与 WAL 尾部

快照里只包含已经完整提交的逻辑会话和当前 `seq/head_hash`。保存采用**同一机器上的临时文件、fsync、`os.replace`、目录fsync**，避免把部分 JSON 当作完整快照。

恢复步骤：

1. 全量扫描并验证 WAL（即使存在快照也执行）。
2. 验证快照摘要、版本、权重身份、seq 和**完整语义前缀**。
3. 从快照状态应用后续 WAL 事件。
4. 从活动会话中的原始 token ID 重算所有 KV tensor。
5. 逐个活动会话与同权重模型的完整前向比较。

因为当前实现始终全量读取/验证 WAL，**不能声称这份快照优化了磁盘恢复速度**；它重点演示逻辑检查点与尾部状态恢复。

## 5. 恢复副本必须重新计算真正的增量 KV

每个 `Replica` 包含独立的 `PagedKVPool`、`CachedModelRunner` 和本地 `DurableJournal`。复制到备机的是同一条已验证的事件记录，而不是直接从主机复制 Python 对象引用。

如果某个 token 在 WAL 中已经提交，但是重建时同一模型的贪心输出不同，恢复会**失败关闭**。这要求模型权重、tokenizer、位置、解码方式与生成状态完全相容。真实采样模式还需要保存 RNG 等状态；本课只允许确定性贪心。

In [9]:
def artifact_identity(tok,models:dict[str,TinyChatLM]) -> dict:
    require(type(models) is dict and set(models)=={'stable','canary'},'模型版本集合不完整')
    configs=[asdict(m.cfg) for m in models.values()]
    require(configs[0]==configs[1],'两个版本配置不同')
    return {'tokenizer':digest(tok.contract()),'weights':{k:state_hash(m.state_dict())
              for k,m in sorted(models.items())},'config':configs[0],
              'max_length':configs[0]['max_length']}


class Replica:
    def __init__(self,name:str,folder:Path,tok,models,capacity_blocks=64,
                 block_size=4,load_existing=False):
        require(name in {'A','B'},'副本名必须A/B')
        self.name=name;self.tok=tok;self.models=copy.deepcopy(models)
        self.identity=artifact_identity(tok,self.models)
        self.folder=Path(folder)
        self.journal=DurableJournal(self.folder,tok,self.identity,load_existing)
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(capacity_blocks,block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(tok,self.models,self.pool)
        self.up=True
        self.last_recovery={'wal_records_verified':0,'snapshot_used':False,
                            'tail_events_semantically_applied':0}
        if load_existing:self.recover_cache()

    @property
    def ledger(self):return self.journal.ledger

    def _expected(self,rid):
        require(rid in self.pool.sessions and rid in self.runner.last_logits,
                '活动请求没有对应真实KV；拒绝偷偷从空上下文解码')
        logits=self.runner.last_logits[rid]
        require(bool(torch.isfinite(logits).all()),'KV输出异常')
        return int(logits.argmax())

    def prevalidate(self,row:dict) -> None:
        require(self.up,'实例已宕机')
        require(artifact_identity(self.tok,self.models)==self.identity,
                '运行模型权重漂移，拒绝生成/应用新事件')
        test=copy.deepcopy(self.ledger);test.apply(row)
        kind=row['kind'];payload=row['payload']
        if kind=='token':
            rid=payload['rid']
            require(rid in self.pool.sessions and rid in self.runner.last_logits,
                    '副本缺失活动KV：先恢复再继续')
            require(self._expected(rid)==payload['token'],
                    '复制到另一个副本时增量输出不一致：权重/前缀/位置存在漂移')

    def accept(self,row:dict) -> None:
        self.prevalidate(row)
        self.journal.append_row(row)
        p=row['payload']
        try:
            if row['kind']=='create':
                self.runner.open_prefill(p['rid'],p['version'],p['prompt_ids'])
            else:
                if p['terminal'] is None:self.runner.step(p['rid'],p['token'])
                else:self.runner.close(p['rid'])
            self.pool.audit()
        except Exception:
            # 已持久事件不能撒谎称“未发生”；该副本必须标记不可用并离线重建。
            self.up=False
            raise

    def create(self,payload:dict) -> dict:
        record=make_record(self.ledger,'create',payload)
        self.accept(record)
        return record

    def token(self,payload:dict) -> dict:
        record=make_record(self.ledger,'token',payload)
        self.accept(record)
        return record

    def recover_cache(self) -> dict:
        require(artifact_identity(self.tok,self.models)==self.identity,
                '恢复实例权重身份不匹配')
        details=self.journal.recover()
        # 丢弃所有易失KV，依据逻辑会话从最早提示重建。
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(self.pool.capacity_blocks,self.pool.block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(self.tok,self.models,self.pool)
        total=0;active=0
        for rid,state in sorted(self.ledger.sessions.items()):
            if state.terminal is not None:continue
            active+=1
            self.runner.open_prefill(rid,state.version,state.prompt_ids)
            total+=len(state.prompt_ids)
            for t in state.generated:
                require(self._expected(rid)==t,
                        '恢复后同一前缀的贪心输出不同，拒绝身份/模型漂移')
                self.runner.step(rid,t)
                total+=1
            self.runner.assert_consistent(rid)
        self.up=True
        details.update({'active_sessions_rebuilt':active,'tokens_recomputed':total,
                        'kv_tensors_restored_from_disk':False,'kv_live_blocks':self.pool.usage()['live_blocks']})
        self.last_recovery=details
        return details

    def audit(self):
        active={k for k,v in self.ledger.sessions.items() if v.terminal is None}
        require(set(self.pool.sessions)==active and set(self.runner.last_logits)==active,
                '逻辑活跃会话与KV实体不一致')
        max_diff=0.0
        for rid in sorted(active):
            st=self.ledger.sessions[rid]
            require(self.pool.sessions[rid].tokens==st.prompt_ids+st.generated,
                    'KV与已确认的逻辑token历史不一致')
            require(self.pool.sessions[rid].version==st.version,'KV与持久版本绑定不一致')
            max_diff=max(max_diff,self.runner.assert_consistent(rid))
        u=self.pool.audit()
        return {'replica':self.name,'logical_seq':self.ledger.seq,'logical_requests':len(self.ledger.sessions),
                'active_sessions':len(active),'kv_live_blocks':u['live_blocks'],
                'max_cache_full_logits_diff':max_diff,'models':self.identity['weights']}


def _request_payload(tok,tenant,key,version,text,max_new_tokens):
    rid=make_rid(tenant,key)
    require(version in {'stable','canary'},'模型版本非法')
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=4,'输出预算非法')
    prompt=tok.apply_chat_template([{'role':'user','content':text}],tokenize=True,
                                   add_generation_prompt=True)
    return {'rid':rid,'tenant':tenant,'key':key,'version':version,
            'prompt_ids':prompt,'max_new_tokens':max_new_tokens}

In [10]:
# 两个物理KV tensor的存储指针不同；模型参数存储也不同。
with tempfile.TemporaryDirectory() as temporary:
    r_a = Replica('A',Path(temporary)/'A',tok,models)
    r_b = Replica('B',Path(temporary)/'B',tok,models)
    print('KV数组内存指针不同:',r_a.pool.data.data_ptr()!=r_b.pool.data.data_ptr())
    print('模型权重对象内存指针不同:',
          r_a.models['stable'].embed.weight.data_ptr()!=r_b.models['stable'].embed.weight.data_ptr())
    print('每token逻辑KV字节:',r_a.pool.bytes_per_token)
    print('每块实际KV字节:',r_a.pool.bytes_per_block)
    print('当前已分配的块数:',r_a.pool.usage()['live_blocks'])

KV数组内存指针不同: True
模型权重对象内存指针不同: True
每token逻辑KV字节: 128
每块实际KV字节: 512
当前已分配的块数: 0


## 6. 同步确认 vs 异步确认：最重要的恢复保证差异

**同步 ACK：** 主实例 WAL fsync 后，必须等备实例验证并 fsync 相同记录，控制器才记 ACK。缺少备实例时，可能出现“主机已写盘、但客户端尚未收到确认”的不确定状态，此时禁止虚构成功。

**异步 ACK：** 主实例 fsync 后就可以回复，备实例可能在最后一个已确认事件之后落后。接管会丢失这些**客户端以为成功的事件**。因此默认阻断，只有演练中显式授权有数据丢失的接管，才允许继续。

RPO 是时间/恢复点差目标，不是“允许丢几条日志”的另一个名字。为了演算，本课定义**每条WAL事件推进一个逻辑 tick**，所以两个差值在本例数值恰好相同，但仍分别报告。

RTO 是恢复到服务可用所需时间目标。本课只计算：

```text
2个控制开销tick
+ ceil(校验的WAL记录数 / 每tick可校验记录数)
+ ceil(重算的KV token数 / 每tick可重算token数)
```

这只是可验证的**成本模型**，不是实测恢复秒数，也不构成生产RTO证据。[AWS的RPO/RTO定义](https://docs.aws.amazon.com/wellarchitected/latest/reliability-pillar/disaster-recovery-dr-objectives.html)

In [11]:
class SimulatedCrash(RuntimeError):
    """明确模拟写盘之后、客户端收到ACK之前的中断，不执行真实进程杀死。"""


def save_model_artifacts(path:Path,tok,models) -> dict:
    require(not Path(path).exists(),'模型artifact不可覆盖')
    identity=artifact_identity(tok,models)
    blob={'version':VERSION,'identity':identity,'tokenizer':tok.contract(),
          'weights':{version:{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                     for version,model in models.items()}}
    torch.save(blob,path)
    return identity


def load_model_artifacts(path:Path):
    # 仅加载本课自己生成、可信来源的纯张量与dict；不执行外部网络下载。
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'artifact版本不匹配')
    tok=ByteMergeChatTokenizer()
    require(blob['tokenizer']==tok.contract(),'tokenizer/chat template身份不匹配')
    identity=blob['identity']
    cfg=ToyConfig(**identity['config'])
    models={}
    for name,values in blob['weights'].items():
        model=inject_lora(TinyChatLM(cfg),rank=4,alpha=8.0)
        model.load_state_dict(values,strict=True)
        models[name]=model.eval().requires_grad_(False)
    require(artifact_identity(tok,models)==identity,'模型artifact哈希不匹配')
    return tok,models


class Coordinator:
    """一个教学用串行控制器，持久化epoch与ACK审计；不是分布式共识。"""
    def __init__(self,root:Path,tok,models,policy='sync',seed=8101,
                 capacity_blocks=64,load_existing=False):
        require(policy in {'sync','async'},'确认模式只能sync/async')
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.control_path=self.root/'control.json'
        self.artifact_path=self.root/'models.pt'
        self.tok=tok;self.models=models
        self.model_identity=artifact_identity(tok,models)
        self.capacity_blocks=capacity_blocks
        self.seed=seed
        if load_existing:
            self.control=self._read_control()
            require(self.control['model_identity']==self.model_identity,
                    '重载模型与原先已确认事件的权重/词表身份不符')
            require(self.control['policy']==policy,'控制器确认策略不可暗中更改')
        else:
            require(not self.control_path.exists() and not self.artifact_path.exists(),
                    '根目录已包含集群持久状态，禁止覆盖')
            save_model_artifacts(self.artifact_path,tok,models)
            self.control={'version':VERSION,'active':'A','epoch':1,'policy':policy,
                          'model_identity':self.model_identity,'ack_seq':0,'ack_tick':0,
                          'ack_history':[],'failovers':[],'quarantined':[],'seed':seed,
                          'capacity_blocks':capacity_blocks}
            atomic_json(self.control_path,self.control)
        self.rep={name:Replica(name,self.root/name,tok,models,capacity_blocks,
                               load_existing=load_existing) for name in ('A','B')}
        self.updates_acknowledged=0
        for name in self.control['quarantined']:
            require(name!=self.control['active'],'活跃主不能被隔离')
            self.rep[name].shutdown()
        self._check_artifacts()

    def _read_control(self):
        try:state=json.loads(self.control_path.read_text(encoding='utf-8'))
        except (UnicodeDecodeError,json.JSONDecodeError,FileNotFoundError) as exc:
            raise LessonError('控制指针无法读取') from exc
        require(type(state) is dict and
                set(state)=={'version','active','epoch','policy','model_identity','ack_seq',
                             'ack_tick','ack_history','failovers','quarantined','seed','capacity_blocks'},
                '控制记录字段非法')
        require(state['version']==VERSION and state['active'] in {'A','B'} and
                type(state['epoch']) is int and state['epoch']>=1,'控制代际非法')
        require(type(state['ack_seq']) is int and state['ack_seq']>=0 and
                state['ack_tick']==state['ack_seq'],'ACK水位非法')
        require(type(state['quarantined']) is list and
                set(state['quarantined'])<={'A','B'} and
                state['active'] not in state['quarantined'],
                '隔离副本列表非法')
        return state

    def _check_artifacts(self):
        token,models=load_model_artifacts(self.artifact_path)
        require(artifact_identity(token,models)==self.model_identity,
                '模型artifact与当前控制身份不符')
        for r in self.rep.values():
            require(r.identity==self.model_identity,'副本模型身份漂移')
            require(artifact_identity(r.tok,r.models)==r.identity,'运行中模型权重被改写')

    def _fence(self,epoch:int):
        durable=self._read_control()
        require(type(epoch) is int and epoch==durable['epoch'] and
                durable['active']==self.control['active'] and
                durable['epoch']==self.control['epoch'],
                'epoch过期或控制指针已被其他控制器切换；拒绝陈旧主节点写入')
        require(self.rep[durable['active']].up,'活跃副本不可用')
        return self.rep[durable['active']]

    def _standby(self):return self.rep['B' if self.control['active']=='A' else 'A']

    def replicate_pending(self):
        primary=self.rep[self.control['active']];target=self._standby()
        require(target.name not in self.control['quarantined'],
                '备机日志已隔离，必须显式丢弃旧分叉并重建')
        require(target.up,'备机不可用，不能确认同步复制')
        require(target.identity==primary.identity,'复制副本模型身份不相等')
        left=target.journal.records;right=primary.journal.records
        require(len(left)<=len(right) and
                all(a==b for a,b in zip(left,right)),
                '日志发生分叉：不可自动拼接，必须隔离并人工处理')
        count=0
        for row in right[len(left):]:
            target.accept(row);count+=1
        require(target.ledger.last_hash==primary.journal.records[target.ledger.seq-1]['hash']
                if target.ledger.seq else target.ledger.last_hash==ZERO_HASH,
                '复制完成后head hash不一致')
        return count

    def _ack(self,row:dict):
        require(type(row) is dict and 'seq' in row and 'hash' in row,'ACK没有WAL证据')
        primary=self.rep[self.control['active']]
        require(primary.ledger.seq>=row['seq'] and
                primary.journal.records[row['seq']-1]==row,'主机日志和ACK不匹配')
        already_acked=row['seq']<=self.control['ack_seq']
        if already_acked:
            require(any(e['seq']==row['seq'] and e['hash']==row['hash']
                        for e in self.control['ack_history']),
                    '历史ACK证据与当前日志不匹配')
        elif self.control['policy']=='sync':
            standby=self._standby()
            require(standby.name not in self.control['quarantined'] and
                    standby.up and standby.ledger.seq>=row['seq'] and
                    standby.journal.records[row['seq']-1]==row,'同步ACK前备机没有持久化该事件')
        if row['seq']>self.control['ack_seq']:
            require(row['seq']==self.control['ack_seq']+1,'不能越过未确认的先前事件')
            self.control['ack_seq']=row['seq'];self.control['ack_tick']=row['tick']
            self.control['ack_history'].append({'seq':row['seq'],'hash':row['hash'],
                                               'epoch':self.control['epoch']})
            atomic_json(self.control_path,self.control)
            self.updates_acknowledged+=1
        return {'seq':row['seq'],'hash':row['hash'],'epoch':self.control['epoch'],
                'durability':'primary+standby' if self.control['policy']=='sync' else 'primary_only'}

    def _commit(self,row:dict,crash_after_durable=False):
        primary=self.rep[self.control['active']]
        primary.accept(row)   # 先写盘fsync，再响应或复制。
        if self.control['policy']=='sync':self.replicate_pending()
        if crash_after_durable:
            raise SimulatedCrash('WAL已fsync但控制ACK未写入；客户端必须同键对账重试')
        return self._ack(row)

    def start(self,epoch,tenant,key,version='stable',text='hello',max_new_tokens=4,
              crash_after_durable=False):
        active=self._fence(epoch)
        payload=_request_payload(self.tok,tenant,key,version,text,max_new_tokens)
        rid=payload['rid']
        if rid in active.ledger.sessions:
            item=active.ledger.sessions[rid]
            require(item.identity()=={k:payload[k] for k in item.identity()},
                    '同一幂等键与先前请求内容/版本不一致')
            row=next(r for r in active.journal.records if r['kind']=='create'
                     and r['payload']['rid']==rid)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'new_event':False,'ack':ack}
        row=make_record(active.ledger,'create',payload)
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'new_event':True,'ack':ack}

    def step(self,epoch,rid,expected_index:int,crash_after_durable=False):
        active=self._fence(epoch)
        require(rid in active.ledger.sessions,'请求不存在')
        item=active.ledger.sessions[rid]
        require(type(expected_index) is int and expected_index>=0,'续传序号非法')
        if expected_index<len(item.generated):
            row=next(r for r in active.journal.records if r['kind']=='token' and
                     r['payload']['rid']==rid and r['payload']['index']==expected_index)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'index':expected_index,'token':row['payload']['token'],
                    'terminal':row['payload']['terminal'],'new_event':False,'ack':ack}
        require(expected_index==len(item.generated),'不能跳过尚未产生的token')
        require(item.terminal is None,'请求已经终止')
        tok=active._expected(rid)
        stop='eos' if tok==self.tok.eos_token_id else (
             'length' if expected_index+1==item.max_new_tokens else None)
        row=make_record(active.ledger,'token',{'rid':rid,'index':expected_index,
                                              'token':tok,'terminal':stop})
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'index':expected_index,'token':tok,'terminal':stop,
                'new_event':True,'ack':ack}

    def checkpoint(self):
        return {name:self.rep[name].journal.checkpoint() for name in ('A','B')}

    def promote(self,epoch,target='B',allow_data_loss=False,rpo_budget_ticks=0,
                rto_budget_ticks=12,tokens_per_tick=32,wal_records_per_tick=16):
        require(type(allow_data_loss) is bool,'数据丢失授权必须显式bool')
        require(target in self.rep and target!=self.control['active'],'接管目标非法')
        persisted=self._read_control()
        require(type(epoch) is int and epoch==self.control['epoch'] and
                persisted==self.control, '接管用epoch过期或控制状态发生变化')
        # 故障后原主机可down；只有这个教学控制指针继续提供围栏，不由原主授权。
        require(all(type(v) is int and v>=0 for v in (rpo_budget_ticks,rto_budget_ticks))
                and tokens_per_tick>0 and wal_records_per_tick>0,'恢复目标/预算非法')
        old_active=self.control['active'];standby=self.rep[target]
        require(standby.up,'备机不可用')
        self._check_artifacts()
        # 在新对象中只从本地持久日志及快照恢复，易失KV不复制过来。
        fresh=Replica(target,self.root/target,self.tok,self.models,self.capacity_blocks,
                      load_existing=True)
        require(artifact_identity(fresh.tok,fresh.models)==self.model_identity,
                '灾难恢复模型/模板指纹不一致')
        audit=fresh.audit()
        ack_seq=self.control['ack_seq']
        lost=max(0,ack_seq-fresh.ledger.seq)
        require(fresh.ledger.seq<=self.rep[old_active].ledger.seq,
                '备机不能凭空比主机多出旧epoch事件')
        # 完整一致的确认前缀不能只是seq相等；还要比较哈希链前缀。
        primary_records=self.rep[old_active].journal.records
        require(fresh.journal.records==primary_records[:len(fresh.journal.records)],
                '备机历史并非旧主已提交前缀；禁止接管分叉日志')
        rpo_gap=max(0,self.control['ack_tick']-fresh.ledger.tick)
        rto=2+math.ceil(fresh.last_recovery['wal_records_verified']/wal_records_per_tick)+(
             math.ceil(fresh.last_recovery['tokens_recomputed']/tokens_per_tick))
        if (lost>0 or rpo_gap>rpo_budget_ticks) and not allow_data_loss:
            raise LessonError('未复制的已ACK事件存在，自动接管必须阻断并报告RPO风险')
        if rto>rto_budget_ticks:
            raise LessonError('预计恢复逻辑tick超出RTO门槛；本课拒绝自动接管')
        report={'from':old_active,'to':target,'old_epoch':epoch,'new_epoch':epoch+1,
                'sync_mode':self.control['policy'],'ack_seq_before':ack_seq,
                'target_seq':fresh.ledger.seq,'lost_acked_events':lost,
                'rpo_gap_logical_ticks':rpo_gap,'rpo_objective_ticks':rpo_budget_ticks,
                'rto_simulated_ticks':rto,'rto_objective_ticks':rto_budget_ticks,
                'rto_formula':'2 + ceil(WAL_verified / records_per_tick) + ceil(KV_replay_tokens / tokens_per_tick)',
                'recovered_active_sessions':audit['active_sessions'],
                'recovered_tokens_recomputed':fresh.last_recovery['tokens_recomputed'],
                'data_loss_override':allow_data_loss and lost>0,
                'no_real_time_measurement':True}
        new_control=copy.deepcopy(self.control)
        new_control['active']=target;new_control['epoch']=epoch+1
        new_control['ack_seq']=fresh.ledger.seq;new_control['ack_tick']=fresh.ledger.tick
        if lost>0:
            new_control['ack_history']=[row for row in new_control['ack_history']
                                        if row['seq']<=fresh.ledger.seq]
            new_control['quarantined']=sorted(set(new_control['quarantined'])|{old_active})
        new_control['failovers'].append(copy.deepcopy(report))
        # 本地控制指针fsync后再关闭旧主；失效旧epoch依赖这个唯一可信控制点。
        atomic_json(self.control_path,new_control)
        self.rep[old_active].shutdown()
        standby.shutdown()
        self.rep[target]=fresh
        self.control=new_control
        return report

    def audit(self):
        require(self._read_control()==self.control,'控制内存与持久状态不一致')
        self._check_artifacts()
        active=self.rep[self.control['active']]
        require(active.up,'当前活跃副本不可用')
        return {'active':self.control['active'],'epoch':self.control['epoch'],
                'ack_seq':self.control['ack_seq'], 'artifacts':self.model_identity,
                'active_audit':active.audit(),
                'standby_head':self._standby().ledger.seq}


def _replica_shutdown(self):
    for sid in list(self.pool.sessions):
        if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
        self.runner.close(sid)
    require(self.pool.audit()['live_blocks']==0,'关闭副本仍有物理KV泄漏')
    self.up=False

Replica.shutdown=_replica_shutdown


def restore_controller(root:Path) -> Coordinator:
    root=Path(root)
    tok,models=load_model_artifacts(root/'models.pt')
    control=json.loads((root/'control.json').read_text(encoding='utf-8'))
    result=Coordinator(root,tok,models,policy=control['policy'],seed=control['seed'],
                       capacity_blocks=control['capacity_blocks'],load_existing=True)
    result.audit()
    return result

In [12]:
with tempfile.TemporaryDirectory() as temporary:
    cluster = Coordinator(Path(temporary),tok,models,policy='sync')
    start = cluster.start(1,'course','request1',version='stable',text='hello')
    print('sync第一个ACK:',start['ack'])
    print('主/备WAL已落盘序号:',cluster.rep['A'].ledger.seq,cluster.rep['B'].ledger.seq)
    print('控制记录ACK水位:',cluster.control['ack_seq'])
    first = cluster.step(1,start['rid'],0)
    retry = cluster.step(1,start['rid'],0)
    print('第一token:',first['token'],'重试是否创建新事件:',retry['new_event'])
    print('请求重试后WAL条数:',cluster.rep['A'].ledger.seq)
    print('实际增量KV与完整前向最大差异:',cluster.rep['A'].audit()['max_cache_full_logits_diff'])
    for r in cluster.rep.values():r.shutdown()

sync第一个ACK: {'seq': 1, 'hash': 'aca319b6ac789829ba8d3e8693b15debff8a5c3fc96b10f8020a99805c5b4923', 'epoch': 1, 'durability': 'primary+standby'}
主/备WAL已落盘序号: 1 1
控制记录ACK水位: 1
第一token: 124 重试是否创建新事件: False
请求重试后WAL条数: 2
实际增量KV与完整前向最大差异: 2.384185791015625e-07


## 7. epoch 围栏：本地唯一指针，不是假共识

跨实例切换前，必须防止旧主继续对外确认写入。演练保存 `active=A/B` 和递增 `epoch`，每个请求都重新读取这个**单一可信本地控制文件**，并比对客户端的epoch。

例如：

```text
epoch=1，A为主，B为备
故障切换后epoch=2，B为主
使用epoch=1的请求：拒绝
```

旧主即使仍活着，不能绕过这个控制器进行写入。然而，**两台真正分离的机器没有自动共享原子文件**：在网络分区、多个控制器或时钟异常下，需要共识/外部受信任fencing、租约和身份认证。本课没有实现这些，不能把它称为真正的跨机器线性一致控制面。

## 8. 十组完整故障演练

| 演练 | 要观察什么 |
|---|---|
| 同步确认后主机失效 | 已ACK事件RPO零丢失、请求续接、旧epoch拒绝 |
| 异步副本落后 | 默认阻断；经显式允许才报告丢失ACK，旧主隔离 |
| fsync后但回复前崩溃 | 同幂等键重试必须返回原日志事件 |
| 损坏的快照 | 正确拒绝；操作员显式丢弃快照后从完整WAL恢复 |
| WAL最后半行 | 不静默跳过；从可信主日志明确修复备机 |
| 模型权重漂移 | 同版本名却不同权重，必须拒绝接管 |
| RTO预算过小 | 阻断自动接管，不假装达标 |
| 两份有效但分叉的WAL | 隔离，不自动拼接历史 |
| 同步模式备机离线 | 主机可fsync，但不得回ACK；恢复后同键对账 |
| 快照加WAL尾部 | 校验完整日志，只应用尾部逻辑状态，重算真实KV |

实验运行路径是本地文件下的十个目录，彼此不会共享模型的KV数据或日志状态。

In [13]:
def _advance(c:Coordinator,rid:str,n:int):
    saved=[]
    for _ in range(n):
        st=c.rep[c.control['active']].ledger.sessions[rid]
        if st.terminal is not None:break
        saved.append(c.step(c.control['epoch'],rid,len(st.generated)))
    return saved


def _case_root(root,name):
    directory=Path(root)/name
    directory.mkdir(parents=True,exist_ok=False)
    return directory


def _new_cluster(root,name,tok,models,policy='sync'):
    return Coordinator(_case_root(root,name),tok,models,policy=policy,seed=8101,
                       capacity_blocks=64)


def _consistent_replay(path:Path) -> dict:
    c=restore_controller(path)
    pointer=c.control
    active=c.rep[pointer['active']]
    output={'active':pointer['active'],'epoch':pointer['epoch'],
            'ack_seq':pointer['ack_seq'],'logical_seq':active.ledger.seq,
            'active_sessions':active.audit()['active_sessions'],
            'replica_model_hashes_match':active.identity==c.model_identity,
            'kv_recomputed':active.last_recovery['tokens_recomputed'],
            'last_log_hash':active.ledger.last_hash}
    for r in c.rep.values():r.shutdown()
    return output


def _atomic_bytes(path:Path,raw:bytes) -> None:
    path=Path(path)
    tmp=path.with_name(path.name+'.repair_tmp')
    require(not tmp.exists(),'修复临时文件未清理')
    try:
        with tmp.open('xb') as f:
            f.write(raw);f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        if hasattr(os,'O_DIRECTORY'):
            fd=os.open(str(path.parent),os.O_RDONLY|os.O_DIRECTORY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if tmp.exists():tmp.unlink()


def repair_standby_wal_from_primary(c:Coordinator) -> dict:
    """明确的教学“人工授权”：从仍可读取且有完整确认前缀的主机重新建立备机日志。

    绝不尝试从损坏WAL里猜部分事件。不能用于双主分叉自动合并。
    """
    primary=c.rep[c.control['active']]
    backup=c._standby()
    require(primary.up,'源主机不可用，不能据其WAL修复')
    rows=primary.journal.read_records()
    require(len(rows)>=c.control['ack_seq'],'源WAL缺失已确认事件')
    require(primary.identity==backup.identity,'模型身份不一致，禁止修复')
    # 当前教学控制器串行调用；真实集群应停止并隔离备机写入并使用可信离线副本。
    backup.shutdown()
    if backup.journal.snapshot_path.exists():backup.journal.snapshot_path.unlink()
    raw=primary.journal.path.read_bytes()
    _atomic_bytes(backup.journal.path,raw)
    new=Replica(backup.name,backup.folder,c.tok,c.models,c.capacity_blocks,load_existing=True)
    require(new.ledger.to_json()==primary.ledger.to_json(),'重建后的备机逻辑状态不一致')
    c.rep[backup.name]=new
    if backup.name in c.control['quarantined']:
        c.control['quarantined'].remove(backup.name)
        atomic_json(c.control_path,c.control)
    return {'restored_seq':new.ledger.seq,'source_hash':primary.ledger.last_hash,
            'kind':'explicit_replace_from_verified_primary',
            'not_a_split_brain_merge':True}


def scenario_sync(root,tok,models):
    c=_new_cluster(root,'sync_zero_rpo',tok,models)
    starts=[c.start(1,'team1','alpha','stable',text='hello',max_new_tokens=4),
            c.start(1,'team2','beta','canary',text='answer 42',max_new_tokens=4)]
    rid0=starts[0]['rid'];rid1=starts[1]['rid']
    _advance(c,rid0,2);_advance(c,rid1,1)
    a=c.rep['A'].audit();b=c.rep['B'].audit()
    require(a['logical_seq']==b['logical_seq']==c.control['ack_seq'],
            '同步模式确认后双实例必须达到同一WAL水位')
    require(a['max_cache_full_logits_diff']<2e-5 and b['max_cache_full_logits_diff']<2e-5,
            '真实KV与全前向不一致')
    checkpoint=c.checkpoint()
    assert_same=all(checkpoint[name]['seq']==c.control['ack_seq'] for name in ('A','B'))
    require(assert_same,'同步快照水位不同')
    c.rep['A'].up=False
    report=c.promote(1,'B',rpo_budget_ticks=0,rto_budget_ticks=12)
    require(report['lost_acked_events']==0 and report['rpo_gap_logical_ticks']==0,
            '同步确认仍丢失ACK数据')
    # 网络断开后重复已确认的token，必须返回原事件，不消耗下一token。
    if c.rep['B'].ledger.sessions[rid0].generated:
        oldseq=c.rep['B'].ledger.seq
        retry=c.step(2,rid0,0)
        require(not retry['new_event'] and c.rep['B'].ledger.seq==oldseq,
                '请求重试导致重复持久token')
    else:retry={'new_event':False}
    stale_error=None
    try:c.start(1,'team3','stale',text='hello')
    except LessonError as exc:stale_error=str(exc)
    require(stale_error is not None,'旧epoch仍能执行新的请求')
    # 同步确认必须重新具备备份副本；本演练是进程故障，旧A的持久盘仍在。
    c.rep['A']=Replica('A',c.root/'A',tok,models,load_existing=True)
    require(c.rep['A'].ledger.to_json()==c.rep['B'].ledger.to_json(),
            '恢复备用实例后逻辑状态不一致')
    for rid in (rid0,rid1):
        while c.rep['B'].ledger.sessions[rid].terminal is None:
            _advance(c,rid,1)
    require(c.rep['B'].pool.audit()['live_blocks']==0,'EOS/长度终止后仍泄漏KV')
    restored=_consistent_replay(c.root)
    return {'kind':'sync','initial_ack_seq':report['ack_seq_before'],
            'checkpoint':checkpoint,'old_primary_audit':a,'old_standby_audit':b,
            'failover':report,'retry_was_duplicate':not retry['new_event'],
            'stale_epoch_blocked':True,'terminal_zero_kv':True,
            'cold_process_replay':restored}


def scenario_async(root,tok,models):
    c=_new_cluster(root,'async_detected_loss',tok,models,policy='async')
    start=c.start(1,'team1','resume','stable',text='hello',max_new_tokens=4)
    rid=start['rid'];require(c.replicate_pending()==1,'预置第一次异步复制失败')
    first=_advance(c,rid,2)
    require(len(first)==2,'演练选定随机模型应能生成2个非EOS token')
    gap=c.control['ack_seq']-c.rep['B'].ledger.seq
    require(gap==2,'异步丢失场景未产生预期滞后')
    c.rep['B'].journal.checkpoint()
    c.rep['A'].up=False
    blocked=False
    try:c.promote(1,'B',rpo_budget_ticks=0)
    except LessonError:blocked=True
    require(blocked and c.control['epoch']==1,'RPO缺口不应被自动放行')
    report=c.promote(1,'B',allow_data_loss=True,rpo_budget_ticks=0,rto_budget_ticks=12)
    require(report['lost_acked_events']==gap and report['data_loss_override'],
            '异步明确数据丢失未报告')
    recovered=c.rep['B'].ledger.sessions[rid]
    require(not recovered.generated,'失去的ACK tokens不能虚构恢复成功')
    require(c.control['quarantined']==['A'] and not c.rep['A'].up,
            '异步损失后旧主日志必须隔离，不能当成健康备机')
    repeat=c.step(2,rid,0)
    require(repeat['new_event'],'丢失的ACK可能在恢复后被重新生成，应明确提示应用去重')
    restored=_consistent_replay(c.root)
    require(restored['active']=='B','恢复时错误接纳了旧主')
    return {'kind':'async','lost_acks':gap,'strict_auto_failover_blocked':blocked,
            'forced_failover':report,'old_primary_quarantined':True,
            'client_must_reconcile_possible_duplicate':True,
            'first_regenerated_token':repeat['token'],'cold_process_replay':restored}


def scenario_ambiguous(root,tok,models):
    c=_new_cluster(root,'fsync_before_ack',tok,models)
    rid='team1__ambiguous'
    crashed=False
    try:c.start(1,'team1','ambiguous',text='hello',crash_after_durable=True)
    except SimulatedCrash:crashed=True
    require(crashed and c.control['ack_seq']==0 and c.rep['A'].ledger.seq==1 and
            c.rep['B'].ledger.seq==1,'未产生明确的持久化与ACK空隙')
    retry=c.start(1,'team1','ambiguous',text='hello')
    require(not retry['new_event'] and c.control['ack_seq']==1,'重试必须确认原create')
    crashed2=False
    try:c.step(1,rid,0,crash_after_durable=True)
    except SimulatedCrash:crashed2=True
    require(crashed2 and c.control['ack_seq']==1 and c.rep['A'].ledger.seq==2,
            'token模拟崩溃不在写盘后')
    reread=c.step(1,rid,0)
    require(not reread['new_event'] and c.control['ack_seq']==2,'同token重试不应多写')
    c.checkpoint()
    restored=_consistent_replay(c.root)
    return {'kind':'ambiguous','crash_after_fsync_create':crashed,
            'crash_after_fsync_token':crashed2,'idempotent_retry':True,
            'ack_seq_after_reconciliation':c.control['ack_seq'],
            'cold_process_replay':restored}


def scenario_snapshot(root,tok,models):
    c=_new_cluster(root,'snapshot_corruption',tok,models)
    c.start(1,'team','snap',text='hello')
    c.checkpoint()
    snapshot=c.rep['B'].journal.snapshot_path
    data=json.loads(snapshot.read_text(encoding='utf-8'))
    data['payload']['state']['seq']+=1
    snapshot.write_text(canonical(data)+'\n',encoding='utf-8')
    blocked=False
    try:Replica('B',c.root/'B',tok,models,load_existing=True)
    except LessonError:blocked=True
    require(blocked,'快照损坏没有被拒绝')
    # 显式人工决定放弃快照，仍从全部已验证WAL重建；不静默回退。
    snapshot.unlink()
    fresh=Replica('B',c.root/'B',tok,models,load_existing=True)
    require(fresh.ledger.to_json()==c.rep['A'].ledger.to_json(),
            '删除损坏快照后WAL恢复不一致')
    repaired_head=fresh.ledger.last_hash
    # 即使攻击者重算摘要，也不能通过与WAL的语义前缀比对。
    snap=fresh.journal.checkpoint()
    wrapped=json.loads(snapshot.read_text(encoding='utf-8'))
    wrapped['payload']['state']['sessions']['team__snap']['max_new_tokens']=3
    wrapped['sha256']=digest(wrapped['payload'])
    snapshot.write_text(canonical(wrapped)+'\n',encoding='utf-8')
    spoof_blocked=False
    try:Replica('B',c.root/'B',tok,models,load_existing=True)
    except LessonError:spoof_blocked=True
    require(spoof_blocked,'重算hash后仍有假快照被接受')
    snapshot.unlink()
    c.rep['B']=Replica('B',c.root/'B',tok,models,load_existing=True)
    return {'kind':'snapshot','corrupted_checksum_blocked':blocked,
            'rehashed_fake_snapshot_blocked':spoof_blocked,
            'operator_discards_bad_snapshot_then_wal_replay':True,
            'recovered_head':repaired_head,'snapshot_seq':snap['seq']}


def scenario_torn(root,tok,models):
    c=_new_cluster(root,'torn_wal_repair',tok,models)
    c.start(1,'team','torn',text='hello')
    path=c.rep['B'].journal.path
    with path.open('ab') as f:f.write(b'{"half_event":');f.flush();os.fsync(f.fileno())
    blocked=False
    try:Replica('B',c.root/'B',tok,models,load_existing=True)
    except LessonError:blocked=True
    require(blocked,'WAL末尾断裂不能静默忽略')
    repair=repair_standby_wal_from_primary(c)
    require(c.rep['B'].journal.read_records()==c.rep['A'].journal.read_records(),
            '备机明确修复后日志不一致')
    return {'kind':'torn_wal','torn_line_blocked':blocked,
            'explicit_operator_replacement':repair}


def scenario_drift(root,tok,models):
    c=_new_cluster(root,'model_identity_drift',tok,models)
    c.start(1,'team','drift','canary',text='hello')
    canary=c.rep['B'].models['canary']
    with torch.no_grad():
        next(p for n,p in canary.named_parameters() if n.endswith('.B')).add_(.05)
    blocked=False
    try:c.promote(1,'B')
    except LessonError:blocked=True
    require(blocked and c.control['active']=='A',
            '备机权重漂移后不得接管')
    c.rep['B'].shutdown()
    c.rep['B']=Replica('B',c.root/'B',tok,models,load_existing=True)
    c.rep['A'].up=False
    report=c.promote(1,'B')
    require(report['lost_acked_events']==0,'修复同一模型后应可零损失接管')
    return {'kind':'identity_drift','drift_blocked':blocked,
            'explicit_model_restore_from_local_artifact':True,'after_repair':report}


def scenario_budget(root,tok,models):
    c=_new_cluster(root,'budget_gate',tok,models)
    c.start(1,'team','budget',text='answer 42')
    blocked=False
    try:c.promote(1,'B',rto_budget_ticks=1)
    except LessonError:blocked=True
    require(blocked and c.control['epoch']==1,'恢复逻辑预算不达标仍被自动通过')
    report=c.promote(1,'B',rto_budget_ticks=12)
    return {'kind':'budget','rto_gate_blocked':blocked,'after_explicit_budget':report,
            'not_measured_wall_clock_seconds':True}


def scenario_split(root,tok,models):
    c=_new_cluster(root,'diverged_log_quarantine',tok,models)
    c.start(1,'team','shared',text='hello')
    # 特权故障注入：绕过控制点给备用盘单独追加另一条合法记录。
    bad_payload=_request_payload(tok,'team','orphan','stable','world',4)
    bad_record=make_record(c.rep['B'].ledger,'create',bad_payload)
    c.rep['B'].accept(bad_record)
    require(c.rep['A'].ledger.last_hash!=c.rep['B'].ledger.last_hash,
            '分叉测试没有产生两份不同日志')
    blocked=False
    try:c.promote(1,'B')
    except LessonError:blocked=True
    require(blocked and c.control['epoch']==1,'双主分叉错误接管成功')
    return {'kind':'split_brain','divergence_blocked':blocked,
            'quarantined_without_automatic_merge':True,
            'standby_head':c.rep['B'].ledger.last_hash,'primary_head':c.rep['A'].ledger.last_hash}


def scenario_sync_unavailable(root,tok,models):
    c=_new_cluster(root,'sync_replica_unavailable',tok,models)
    c.rep['B'].up=False
    unavailable=False
    try:c.start(1,'team','pending',text='hello')
    except LessonError:unavailable=True
    require(unavailable and c.control['ack_seq']==0 and c.rep['A'].ledger.seq==1,
            '备机不可用时不得给出sync ACK')
    c.rep['B'].up=True
    retried=c.start(1,'team','pending',text='hello')
    require(not retried['new_event'] and c.control['ack_seq']==1 and
            c.rep['B'].ledger.seq==1,'恢复后同键应补齐复制且确认原请求')
    return {'kind':'sync_unavailable','ack_withheld_without_standby':unavailable,
            'durable_but_not_acked_seq':1,'retried_original_event':True}


def scenario_compare(root,tok,models):
    c=_new_cluster(root,'snapshot_wal_tail',tok,models)
    start=c.start(1,'team','tail',text='hello')
    checkpoint=c.checkpoint()
    before=checkpoint['B']['seq']
    _advance(c,start['rid'],2)
    require(c.rep['B'].ledger.seq>before,'快照后WAL未增长')
    restored=Replica('B',c.root/'B',tok,models,load_existing=True)
    require(restored.last_recovery['snapshot_used'] and
            restored.last_recovery['tail_events_semantically_applied']==
             c.rep['B'].ledger.seq-before,'没有正确复用快照应用WAL尾部')
    require(restored.ledger.to_json()==c.rep['B'].ledger.to_json(),
            '快照+尾部重放与原会话状态不同')
    return {'kind':'snapshot_tail','snapshot_seq':before,
            'replayed_tail_events':restored.last_recovery['tail_events_semantically_applied'],
            'full_wal_integrity_scanned':restored.last_recovery['wal_records_verified'],
            'kv_rebuilt_not_loaded':not restored.last_recovery['kv_tensors_restored_from_disk'],
            'kv_consistent':restored.audit()['max_cache_full_logits_diff']<2e-5}


def run_experiment(root:Path=Path('lesson_81_outputs'),seed=8101):
    require(type(seed) is int and seed>=0,'seed非法')
    torch.set_num_threads(1)
    seed_all(seed)
    tok,models=build_servable_models(seed)
    folder=Path(root)/datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
    folder.mkdir(parents=True,exist_ok=False)
    run_metadata={'version':VERSION,'seed':seed,'python':platform.python_version(),
                  'torch':str(torch.__version__),'device':'cpu',
                  'status':'running','model_identity':artifact_identity(tok,models),
                  'sources':SOURCES,
                  'limitations':['two_instances_as_single_process_on_local_filesystem',
                      'no_network_no_consensus_no_distributed_CAS_no_real_host_fencing',
                      'RPO_RTO_are_logical_ticks_not_seconds',
                      'model_randomly_initialized_not_finetuned_capability',
                      'persisted_token_ids_may_contain_sensitive_data; no_encryption',
                      'only_greedy_deterministic_decoding; no_rng_replay',
                      'checkpoint_is_logical_not_KV_tensor_snapshot']}
    write_json(folder/'manifest.json',run_metadata)
    scenarios={
        'sync_zero_rpo':scenario_sync,
        'async_detected_loss':scenario_async,
        'fsync_before_ack':scenario_ambiguous,
        'snapshot_corruption':scenario_snapshot,
        'torn_wal_repair':scenario_torn,
        'model_identity_drift':scenario_drift,
        'budget_gate':scenario_budget,
        'diverged_log_quarantine':scenario_split,
        'sync_replica_unavailable':scenario_sync_unavailable,
        'snapshot_wal_tail':scenario_compare,
    }
    reports={}
    try:
        for name,fn in scenarios.items():
            result=fn(folder,tok,models)
            reports[name]=result
            print(name,'通过',flush=True)
        write_json(folder/'scenarios.json',reports)
        independent=reload_run(folder)
        write_json(folder/'reload.json',independent)
        run_metadata['status']='completed';write_json(folder/'manifest.json',run_metadata)
        print('输出目录：',folder,flush=True)
        return folder
    except Exception as exc:
        run_metadata['status']='failed';run_metadata['error']=type(exc).__name__+': '+str(exc)
        write_json(folder/'manifest.json',run_metadata)
        raise


def reload_run(folder:Path) -> dict:
    folder=Path(folder)
    manifest=json.loads((folder/'manifest.json').read_text(encoding='utf-8'))
    require(manifest['version']==VERSION and manifest['status'] in {'running','completed'},
            '演练版本或状态非法')
    reports=json.loads((folder/'scenarios.json').read_text(encoding='utf-8'))
    required={'sync_zero_rpo','async_detected_loss','fsync_before_ack','snapshot_corruption',
              'torn_wal_repair','model_identity_drift','budget_gate',
              'diverged_log_quarantine','sync_replica_unavailable','snapshot_wal_tail'}
    require(set(reports)==required,'场景文件缺失')
    results={}
    for name in ('sync_zero_rpo','async_detected_loss','fsync_before_ack',
                 'snapshot_wal_tail','model_identity_drift','budget_gate'):
        # 只有这些已成功完成或仍有合法WAL的场景，允许从文件独立恢复。
        c=restore_controller(folder/name)
        require(c.control['model_identity']==manifest['model_identity'],
                '重载副本模型身份与manifest不同')
        current=c.rep[c.control['active']]
        state=current.audit()
        results[name]={'active':c.control['active'],'epoch':c.control['epoch'],
                       'seq':current.ledger.seq,'active_sessions':state['active_sessions'],
                       'kv_logits_consistent':state['max_cache_full_logits_diff']<2e-5,
                       'kv_rebuilt_from_logical_ids':True}
        for r in c.rep.values():r.shutdown()
    return {'replayed_scenarios':results,'refused_to_auto_replay_corrupt_or_diverged':
            ['diverged_log_quarantine'],
            'scope':'trusted local same-CPU model artifacts and strict WAL replay; no live distributed failover'}

## 9. 先检查，再运行默认完整演练

自动检查从空日志、错误 token/停止条件开始，覆盖 WAL 顺序、损坏哈希、重算摘要的假快照、复制确认、模型漂移、断链、旧epoch、控制状态、真实KV和所有页释放。

检查通过只说明这些固定条件得到覆盖；没有实测真实多节点网络、共识或多故障域一致性。

In [14]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def equal(self,name,a,b):self.ok(name,a==b)
    def close(self,name,a,b,atol=2e-5):
        self.ok(name,torch.allclose(torch.as_tensor(a,dtype=torch.float64),
                                   torch.as_tensor(b,dtype=torch.float64),atol=atol,rtol=2e-5))
    def raises(self,name,fn):
        try:fn()
        except (LessonError,ValueError,SimulatedCrash):
            self.names.append(name);return
        raise AssertionError(name+': 错误条件应显式拒绝')


def run_checks() -> dict:
    torch.set_num_threads(1)
    seed_all(8111)
    c=Checks()
    tok,models=build_servable_models(8101)
    identity=artifact_identity(tok,models)
    c.ok('两个不同模型版本权重有独立hash',identity['weights']['stable']!=identity['weights']['canary'])
    c.equal('模型配置名称固定',set(models),{'stable','canary'})
    c.ok('同一基座可比较',base_hash(models['stable'])==base_hash(models['canary']))
    c.ok('参考模型全部冻结',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('字符分词器版本绑定',len(tok.contract()['merges'])>0)
    c.equal('中文分词器可往返',tok.decode(tok.encode('你好')), '你好')
    c.raises('UTF8半字符不能静默替换',lambda:tok.decode([tok.byte_offset+0xe4]))
    c.raises('非法控制token不能正文decode',lambda:tok.decode([tok.eos_token_id]))
    p=_request_payload(tok,'team','demo','stable','hello',3)
    c.equal('请求ID有租户键',p['rid'],'team__demo')
    c.ok('提示包含明确generation header',len(p['prompt_ids'])>=4)
    c.ok('提示是全整数token',all(type(i) is int for i in p['prompt_ids']))
    for tenant,key in [('a:b','x'),('','x'),('abc','a b')]:
        c.raises('非法幂等键_'+repr((tenant,key)),lambda tenant=tenant,key=key:make_rid(tenant,key))
    c.raises('无法使用未知版本',lambda:_request_payload(tok,'t','k','unseen','hello',2))
    c.raises('不允许空输出预算',lambda:_request_payload(tok,'t','k','stable','hello',0))
    c.raises('过大输出预算',lambda:_request_payload(tok,'t','k','stable','hello',5))
    ledger=Ledger(tok,identity)
    create=make_record(ledger,'create',p)
    c.equal('首个WAL记录seq1',create['seq'],1)
    c.equal('首个WAL前驱是零hash',create['prev'],ZERO_HASH)
    ledger.apply(create)
    c.equal('逻辑事件应用后seq前进',ledger.seq,1)
    c.equal('日志记录与请求key一致',ledger.sessions[p['rid']].version,'stable')
    c.raises('重复create必须拒绝',lambda:ledger.apply(make_record(ledger,'create',p)))
    bad=copy.deepcopy(create);bad['seq']=99
    c.raises('seq篡改拒绝',lambda:Ledger(tok,identity).apply(bad))
    bad=copy.deepcopy(create);bad['hash']='f'*64
    c.raises('内容hash篡改拒绝',lambda:Ledger(tok,identity).apply(bad))
    bad=copy.deepcopy(create);bad['prev']='a'*64;bad['hash']=digest({k:v for k,v in bad.items() if k!='hash'})
    c.raises('前驱hash篡改拒绝',lambda:Ledger(tok,identity).apply(bad))
    bad=copy.deepcopy(create);bad['tick']=5;bad['hash']=digest({k:v for k,v in bad.items() if k!='hash'})
    c.raises('tick不连续拒绝',lambda:Ledger(tok,identity).apply(bad))
    bad=copy.deepcopy(create);bad['payload']['prompt_ids']=[999999];bad['hash']=digest({k:v for k,v in bad.items() if k!='hash'})
    c.raises('非法prompt ID拒绝',lambda:Ledger(tok,identity).apply(bad))
    bad=copy.deepcopy(create);bad['payload']['version']='missing';bad['hash']=digest({k:v for k,v in bad.items() if k!='hash'})
    c.raises('模型版本不存在拒绝',lambda:Ledger(tok,identity).apply(bad))
    normal={'rid':p['rid'],'index':0,'token':tok.byte_offset+ord('a'),'terminal':None}
    token=make_record(ledger,'token',normal)
    ledger.apply(token)
    c.equal('token被保存到逻辑序列',ledger.sessions[p['rid']].generated,[normal['token']])
    c.raises('重复token index拒绝',lambda:make_record(ledger,'token',normal))
    c.raises('错误EOS状态拒绝',lambda:make_record(ledger,'token',
             {'rid':p['rid'],'index':1,'token':tok.eos_token_id,'terminal':None}))
    c.raises('非EOS伪装EOS拒绝',lambda:make_record(ledger,'token',
             {'rid':p['rid'],'index':1,'token':tok.byte_offset+ord('a'),'terminal':'eos'}))
    second=make_record(ledger,'token',{'rid':p['rid'],'index':1,
                                'token':tok.byte_offset+ord('b'),'terminal':None})
    ledger.apply(second)
    last=make_record(ledger,'token',{'rid':p['rid'],'index':2,
                                'token':tok.byte_offset+ord('c'),'terminal':'length'})
    ledger.apply(last)
    c.equal('长度终止被记录',ledger.sessions[p['rid']].terminal,'length')
    c.raises('终态不能继续写入',lambda:make_record(ledger,'token',
             {'rid':p['rid'],'index':3,'token':tok.eos_token_id,'terminal':'eos'}))
    same=Ledger.from_json(tok,identity,ledger.to_json())
    c.equal('逻辑快照往返',same.to_json(),ledger.to_json())
    damaged=ledger.to_json();damaged['sessions'][p['rid']]['generated'].append(tok.eos_token_id)
    c.raises('快照末尾多余EOS拒绝',lambda:Ledger.from_json(tok,identity,damaged))
    damaged=ledger.to_json();damaged['sessions'][p['rid']]['rid']='wrong'
    c.raises('快照会话key错配拒绝',lambda:Ledger.from_json(tok,identity,damaged))
    damaged=ledger.to_json();damaged['sessions'][p['rid']]['version']='unknown'
    c.raises('快照引用未知模型拒绝',lambda:Ledger.from_json(tok,identity,damaged))
    with tempfile.TemporaryDirectory() as td:
        folder=Path(td)
        journal=DurableJournal(folder,tok,identity)
        journal.append_row(create)
        journal.append_row(token)
        c.equal('WAL写入两行',len(journal.records),2)
        c.equal('严格WAL读回行数',len(journal.read_records()),2)
        snap=journal.checkpoint()
        c.equal('快照与WAL末尾一致',snap['seq'],2)
        original=json.loads(journal.snapshot_path.read_text(encoding='utf-8'))
        corrupt=copy.deepcopy(original);corrupt['sha256']='0'*64
        atomic_json(journal.snapshot_path,corrupt)
        c.raises('有效内容但错误快照摘要必须拒绝',
                 lambda:DurableJournal(folder,tok,identity,load_existing=True))
        atomic_json(journal.snapshot_path,original)
        c.ok('快照明确不存储KV张量','NOT raw KV' in snap['contents'])
        recovered=DurableJournal(folder,tok,identity,load_existing=True)
        c.equal('快照恢复seq正确',recovered.ledger.seq,2)
        c.ok('快照恢复不跳过WAL审计',recovered.recover()['wal_records_verified']==2)
        recovered.append_row(second)
        tail=DurableJournal(folder,tok,identity,load_existing=True)
        c.equal('快照后应用1条尾记录',tail.recover()['tail_events_semantically_applied'],1)
        c.equal('快照尾部hash等于全WAL',tail.ledger.last_hash,second['hash'])
        # 伪造快照内容后也重算裸SHA：必须与完整WAL前缀逐字段核对。
        fake=json.loads(journal.snapshot_path.read_text(encoding='utf-8'))
        fake['payload']['state']['sessions'][p['rid']]['max_new_tokens']=4
        fake['sha256']=digest(fake['payload'])
        atomic_json(journal.snapshot_path,fake)
        c.raises('重算SHA的伪快照与WAL前缀不一致',
                 lambda:DurableJournal(folder,tok,identity,load_existing=True))
        # 使用原始已确认的会话状态恢复合法快照，继续后续故障测试。
        journal.checkpoint()
        # 变更snapshot上层可信身份：必须拒绝，即使自行重算本地摘要。
        wrapped=json.loads(journal.snapshot_path.read_text(encoding='utf-8'))
        wrapped['payload']['model_identity']['weights']['stable']='a'*64
        wrapped['sha256']=digest(wrapped['payload'])
        atomic_json(journal.snapshot_path,wrapped)
        c.raises('重写合法hash的模型身份仍拒绝',lambda:DurableJournal(folder,tok,identity,load_existing=True))
        journal.snapshot_path.unlink()
        recovery_no_snapshot=DurableJournal(folder,tok,identity,load_existing=True)
        c.equal('明确丢弃坏快照可用完整WAL恢复',recovery_no_snapshot.ledger.seq,3)
        with journal.path.open('ab') as file:file.write(b'partial{')
        c.raises('WAL尾截断行不能静默跳过',lambda:DurableJournal(folder,tok,identity,load_existing=True))
    with tempfile.TemporaryDirectory() as td:
        root=Path(td)
        c1=Coordinator(root,tok,models,policy='sync')
        a=c1.start(1,'team','same',text='hello')
        c.equal('同步ACK后主日志水位',c1.rep['A'].ledger.seq,1)
        c.equal('同步ACK后备日志水位',c1.rep['B'].ledger.seq,1)
        c.equal('同步模式ACK水位',c1.control['ack_seq'],1)
        c.ok('两份模型对象不共享张量',c1.rep['A'].models['stable'].embed.weight.data_ptr()!=
             c1.rep['B'].models['stable'].embed.weight.data_ptr())
        c.ok('两份真实KV tensor不共享存储',c1.rep['A'].pool.data.data_ptr()!=c1.rep['B'].pool.data.data_ptr())
        c.ok('真实KV每token有实际字节',c1.rep['A'].pool.bytes_per_token>0)
        c.ok('原地真实KV缓存与全上下文相符',c1.rep['A'].audit()['max_cache_full_logits_diff']<2e-5)
        c.equal('同键重试不追加WAL',c1.start(1,'team','same',text='hello')['new_event'],False)
        c.equal('同键重试仍在seq1',c1.control['ack_seq'],1)
        c.raises('同键不同输入必须拒绝',lambda:c1.start(1,'team','same',text='world'))
        c.raises('同键不同模型必须拒绝',lambda:c1.start(1,'team','same',version='canary'))
        c.raises('初始epoch过期拒绝',lambda:c1.start(0,'team','stale'))
        tok0=c1.step(1,a['rid'],0)
        c.equal('首token写入索引0',tok0['index'],0)
        c.equal('首token当前ACK号',c1.control['ack_seq'],2)
        c.equal('重复token返回同一token',c1.step(1,a['rid'],0)['token'],tok0['token'])
        c.equal('重复token不写新日志',c1.rep['A'].ledger.seq,2)
        c.raises('漏掉token序号拒绝',lambda:c1.step(1,a['rid'],2))
        c.ok('两个副本token序列一致',c1.rep['A'].ledger.to_json()==c1.rep['B'].ledger.to_json())
        c.ok('两个增量KV都能对齐full forward',c1.rep['B'].audit()['max_cache_full_logits_diff']<2e-5)
        snapshot=c1.checkpoint()
        c.equal('快照两边seq都相等',snapshot['A']['seq'],snapshot['B']['seq'])
        c.ok('快照实际是JSON',c1.rep['A'].journal.snapshot_path.suffix=='.json')
        checkpoint_bytes=c1.rep['A'].journal.snapshot_path.read_bytes()
        c.ok('KV tensor没有写进快照文件',b'pickle' not in checkpoint_bytes and
              b'PK\x03\x04' not in checkpoint_bytes)
        restored=restore_controller(root)
        c.equal('从模型文件恢复身份',restored.model_identity,c1.model_identity)
        c.equal('从checkpoint恢复ACK水位',restored.control['ack_seq'],2)
        c.ok('恢复后重算KV与全前向匹配',restored.rep['A'].audit()['max_cache_full_logits_diff']<2e-5)
        c.ok('恢复时确实重算过输入',restored.rep['A'].last_recovery['tokens_recomputed']>0)
        c.equal('持久快照使用情况',restored.rep['B'].last_recovery['snapshot_used'],True)
        c.raises('不能重复初始化已占用目录',lambda:Coordinator(root,tok,models))
        fake=copy.deepcopy(models)
        with torch.no_grad():next(p for n,p in fake['stable'].named_parameters() if n.endswith('.B')).add_(.01)
        c.raises('不同权重重载被拒绝',lambda:Coordinator(root,tok,fake,load_existing=True))
        live=c1.rep['A'].models['stable']
        with torch.no_grad():next(p for n,p in live.named_parameters() if n.endswith('.B')).add_(.01)
        c.raises('运行时模型权重漂移拒绝下一事件',
                 lambda:c1.start(1,'team','mutated'))
        _,trusted=load_model_artifacts(root/'models.pt')
        c1.rep['A'].models=copy.deepcopy(trusted)
        c1.rep['A'].runner.models=c1.rep['A'].models
        # 控制代际切换后，同一目录的旧对象也必须在每次写入时重新核对持久指针。
        c1.rep['A'].up=False
        c.raises('低于模型化RTO成本的自动接管必须阻断',
                 lambda:c1.promote(1,'B',rto_budget_ticks=1))
        c.equal('RTO预算阻断后仍保持原epoch',c1.control['epoch'],1)
        outcome=c1.promote(1,'B')
        c.equal('同步模式接管ACK零损失',outcome['lost_acked_events'],0)
        c.equal('同步模式RPO逻辑tick为0',outcome['rpo_gap_logical_ticks'],0)
        c.ok('RTO模拟成本是正数',outcome['rto_simulated_ticks']>=2)
        c.equal('接管后epoch递增',c1.control['epoch'],2)
        c.raises('老进程带旧epoch仍不许写',lambda:restored.start(1,'team','stale'))
        c.raises('旧epoch在新控制器中也拒绝',lambda:c1.start(1,'team','stale'))
        c.ok('主机旧KV已经清理',c1.rep['A'].pool.usage()['live_blocks']==0)
        c.ok('恢复新主完整KV一致',c1.rep['B'].audit()['max_cache_full_logits_diff']<2e-5)
        # 同步确认不能在没有备机情况下继续写新数据。
        c.raises('单备机丢失时新sync写必须拒绝',lambda:c1.start(2,'team','noquorum'))
        c.equal('单备机丢失新sync写未ACK',c1.control['ack_seq'],2)
        c1.rep['A']=Replica('A',root/'A',tok,models,load_existing=True)
        c.equal('恢复standby后可以补充同步',c1.replicate_pending(),1)
        recovered_ack=c1.start(2,'team','noquorum')
        c.equal('同键重试确认原WAL',recovered_ack['new_event'],False)
        c.equal('恢复备机后ACK更新',c1.control['ack_seq'],3)
        c.ok('同步确认两边head一致',c1.rep['A'].ledger.last_hash==c1.rep['B'].ledger.last_hash)
        for replica in c1.rep.values():replica.shutdown()
        c.ok('主动关闭后的物理块清零',all(replica.pool.usage()['live_blocks']==0 for replica in c1.rep.values()))
        c.ok('已释放的实际KV数据为零',all(not bool(replica.pool.data.count_nonzero()) for replica in c1.rep.values()))
    with tempfile.TemporaryDirectory() as td:
        c2=Coordinator(td,tok,models,policy='async')
        c2.start(1,'team','gap',text='hello')
        c.equal('异步确认时备机可以为0',c2.rep['B'].ledger.seq,0)
        c.equal('异步ACK已经前进',c2.control['ack_seq'],1)
        c.raises('没有RPO允许不能接管',lambda:c2.promote(1,'B'))
        c.equal('被阻止后控制epoch不动',c2.control['epoch'],1)
        withloss=c2.promote(1,'B',allow_data_loss=True)
        c.equal('操作员显式丢弃一个ACK事件',withloss['lost_acked_events'],1)
        c.equal('RPO按逻辑tick记为1',withloss['rpo_gap_logical_ticks'],1)
        c.equal('数据丢失覆盖写入报告',withloss['data_loss_override'],True)
        c.ok('备用WAL为空时恢复为空服务',c2.rep['B'].ledger.seq==0 and not c2.rep['B'].ledger.sessions)
        c.equal('接管后唯一活动主为B',c2.control['active'],'B')
        c.equal('有数据丢失后旧主被隔离',c2.control['quarantined'],['A'])
        c.raises('隔离副本不能当作正常复制目标',c2.replicate_pending)
        c.ok('隔离的旧主内存KV已清空',c2.rep['A'].pool.usage()['live_blocks']==0)
    with tempfile.TemporaryDirectory() as td:
        fork=Coordinator(td,tok,models)
        fork.start(1,'team','forksource',text='hello')
        foreign=make_record(fork.rep['B'].ledger,'create',
                            _request_payload(tok,'team','forked','stable','world',4))
        fork.rep['B'].accept(foreign)
        c.ok('分叉日志即使本地hash合法也具有不同前缀',
             fork.rep['B'].ledger.last_hash!=fork.rep['A'].ledger.last_hash)
        c.raises('拒绝把分叉日志直接接管',lambda:fork.promote(1,'B'))
    # 简单计算反例：事件数量与RPO时间不是一个指标，只在本课1事件/1tick时数值相等。
    c.equal('RPO定义为ACK时钟与恢复点差',max(0,12-10),2)
    c.equal('RTO预算不满足时明确不通过',4<=3,False)
    for name in ('sync_zero_rpo','async_detected_loss','fsync_before_ack'):
        c.ok('核心演练函数可定位_'+name,callable({
            'sync_zero_rpo':scenario_sync,'async_detected_loss':scenario_async,
            'fsync_before_ack':scenario_ambiguous}[name]))
    return {'passed':len(c.names),'checks':c.names,'scope':
            'single-process CPU PyTorch real KV, fsynced local WAL, no network consensus'}

In [15]:
checks = run_checks()
print('固定检查项:',checks['passed'])
print('检查范围:',checks['scope'])

固定检查项: 118
检查范围: single-process CPU PyTorch real KV, fsynced local WAL, no network consensus


In [16]:
RUN_DIR = None
if RUN_FULL_EXPERIMENT:
    RUN_DIR = run_experiment(OUTPUT_ROOT,SEED)
    write_json(RUN_DIR/'checks.json',checks)
else:
    print('完整演练已关闭；没有生成模拟测试成绩。')

sync_zero_rpo 通过


async_detected_loss 通过


fsync_before_ack 通过


snapshot_corruption 通过


torn_wal_repair 通过


model_identity_drift 通过


budget_gate 通过


diverged_log_quarantine 通过


sync_replica_unavailable 通过


snapshot_wal_tail 通过


输出目录： lesson_81_outputs/run_20261105T071151_575947Z


## 10. 实际演练结果如何读？

这里读取你刚才运行生成的真实 `scenarios.json`，不是人工编造结果。默认单CPU种子8101实验的关键数值如下：

- 同步接管前已确认5条事件，恢复点与最后ACK一致：**丢失0条ACK事件，RPO差0逻辑tick**。
- 同步接管需要重算18个KV token，**模拟RTO成本4逻辑tick**；这不是4秒。
- 异步场景备实例只收到1条事件，主实例已ACK3条；严格接管被阻止，显式数据丢失接管报告**丢失2条ACK事件、RPO差2逻辑tick**。
- 日志分叉不做自动合并；旧主的额外历史不能假定与新主顺序兼容。

若改变种子、代码或硬件，应该以实际生成文件为准。

In [17]:
if RUN_DIR is not None:
    results = json.loads((RUN_DIR/'scenarios.json').read_text(encoding='utf-8'))
    sync = results['sync_zero_rpo']['failover']
    async_run = results['async_detected_loss']['forced_failover']
    print('同步：', {k:sync[k] for k in ('ack_seq_before','lost_acked_events',
          'rpo_gap_logical_ticks','rto_simulated_ticks','recovered_tokens_recomputed')})
    print('异步：', {k:async_run[k] for k in ('ack_seq_before','target_seq','lost_acked_events',
          'rpo_gap_logical_ticks','data_loss_override')})
    print('版本漂移阻断:',results['model_identity_drift']['drift_blocked'])
    print('WAL断尾拒绝:',results['torn_wal_repair']['torn_line_blocked'])
    print('合法hash但状态不符的快照拒绝:',results['snapshot_corruption']['rehashed_fake_snapshot_blocked'])
    print('快照后逻辑事件重放数:',results['snapshot_wal_tail']['replayed_tail_events'])

同步： {'ack_seq_before': 5, 'lost_acked_events': 0, 'rpo_gap_logical_ticks': 0, 'rto_simulated_ticks': 4, 'recovered_tokens_recomputed': 18}
异步： {'ack_seq_before': 3, 'target_seq': 1, 'lost_acked_events': 2, 'rpo_gap_logical_ticks': 2, 'data_loss_override': True}
版本漂移阻断: True
WAL断尾拒绝: True
合法hash但状态不符的快照拒绝: True
快照后逻辑事件重放数: 2


## 11. 失败与接管不等于精确一次投递

在 `fsync_before_ack` 场景，WAL已提交但控制ACK尚未记录。客户端不能简单判断请求没有执行：必须用相同幂等键和事件序号对账重试。若明确收到客户端响应后才提交ACK的外部系统发生故障，端到端**exactly-once**需要额外协议或幂等副作用设计。

异步场景更复杂：旧主已经ACK过的 token 可能不在新主WAL中。重新生成可能输出同一 token，但**不能用“文本恰好相同”证明没有重复下游副作用**。本课在明确的数据丢失接管中隔离旧主日志，不自动把旧主当作备实例重新加入。

In [18]:
if RUN_DIR is not None:
    r=results['fsync_before_ack']
    print('落盘后ACK前两次模拟崩溃:',r['crash_after_fsync_create'],r['crash_after_fsync_token'])
    print('重试使用原始事件:',r['idempotent_retry'])
    r=results['async_detected_loss']
    print('异步遗失ACK:',r['lost_acks'])
    print('旧主隔离:',r['old_primary_quarantined'])
    print('客户端可能需要去重与人工对账:',r['client_must_reconcile_possible_duplicate'])

落盘后ACK前两次模拟崩溃: True True
重试使用原始事件: True
异步遗失ACK: 2
旧主隔离: True
客户端可能需要去重与人工对账: True


## 12. 保存了什么？没有保存什么？

每个场景目录包含：

```text
models.pt             可信本地随机模型的权重张量与身份
control.json          单点模拟epoch、活动实例、ACK与隔离记录
A/events.jsonl        副本A自己的持久事件
A/snapshot.json       可选逻辑状态快照
B/events.jsonl        副本B自己的持久事件
B/snapshot.json       可选逻辑状态快照
```

根目录有 `manifest.json`、`scenarios.json`、`checks.json` 和 `reload.json`。

快照**没有** `torch.save` 的 KV tensor、块指针、真实远端租约或进程内隐式状态。重载运行时使用 `torch.load(..., weights_only=True)` 读取本课可信模型，然后以 WAL 中逻辑token重新prefill。不能拿陌生 `.pt` 文件测试，也不能把这份本地文件系统控制器替换为真实集群控制面。

In [19]:
if RUN_DIR is not None:
    snap=RUN_DIR/'sync_zero_rpo'/'B'/'snapshot.json'
    data=json.loads(snap.read_text(encoding='utf-8'))
    print('快照字段:',list(data))
    print('快照payload字段:',list(data['payload']))
    print('快照逻辑会话字段:',list(data['payload']['state']['sessions'].values())[0].keys())
    print('是否出现原始KV tensor字段:', 'kv_tensor' in canonical(data))
    print('模型文件存在:',(RUN_DIR/'sync_zero_rpo'/'models.pt').is_file())

快照字段: ['payload', 'sha256']
快照payload字段: ['model_identity', 'state', 'version']
快照逻辑会话字段: dict_keys(['generated', 'key', 'max_new_tokens', 'prompt_ids', 'rid', 'tenant', 'terminal', 'version'])
是否出现原始KV tensor字段: False
模型文件存在: True


## 13. 独立重载与快照一致性

下面重新创建所有合法场景中的模型与副本对象，校验原文件身份、WAL、快照与缓存重算结果。分叉文件不能默认执行无条件合并。

跨设备PyTorch浮点运算未保证 bitwise 相等，默认仅在同CPU环境验证。这里没有时间性能基准，也没有声称真实RPO秒数。

In [20]:
if RUN_DIR is not None:
    restored_report = reload_run(RUN_DIR)
    print('独立恢复的合法演练数:',len(restored_report['replayed_scenarios']))
    for name, item in restored_report['replayed_scenarios'].items():
        print(name,item['active'],item['seq'],'cache_full一致=',item['kv_logits_consistent'])
    print('分叉自动合并状态:',restored_report['refused_to_auto_replay_corrupt_or_diverged'])

独立恢复的合法演练数: 6
sync_zero_rpo B 10 cache_full一致= True
async_detected_loss B 2 cache_full一致= True
fsync_before_ack A 2 cache_full一致= True
snapshot_wal_tail A 3 cache_full一致= True
model_identity_drift B 1 cache_full一致= True
budget_gate B 1 cache_full一致= True
分叉自动合并状态: ['diverged_log_quarantine']


## 14. 独立练习：先填空，再看参考答案

练习区与后面参考答案使用不同函数名称。开关默认 False，避免未填完的函数阻断完整演练。

### 练习 A：RPO 与丢失 ACK 的区别

实现 `rpo_report_ex(last_ack_tick, recovered_tick, last_ack_seq, recovered_seq, budget_ticks)`，返回：

```python
{'rpo_gap_ticks': ..., 'lost_acked_events': ..., 'within_budget': ...}
```

时间与事件计数单独计算，拒绝负数和恢复点大于ACK时钟等非法输入。

In [21]:
def check_exercise_a(fn):
    value=fn(15,12,9,6,3)
    require(value=={'rpo_gap_ticks':3,'lost_acked_events':3,'within_budget':True},
            '练习A：逻辑RPO/丢失ACK数量不符')
    value=fn(15,12,9,6,2)
    require(not value['within_budget'],'练习A：严格超预算未拒绝')
    for args in ((2,3,2,3,0),(5,4,2,3,0),(5,3,2,2,-1)):
        try:fn(*args)
        except (ValueError,LessonError):pass
        else:raise AssertionError('练习A：不合法时钟/seq/预算未拒绝')
    return {'exercise':'A','passed':True}

def check_exercise_b(fn):
    require(fn(3,3,'B','B') is True,'练习B：正确epoch应允许')
    for args in ((2,3,'B','B'),(3,3,'A','B'),(4,3,'B','B')):
        try:fn(*args)
        except (ValueError,LessonError):pass
        else:raise AssertionError('练习B：旧/未授权epoch被接受')
    try:fn(True,3,'B','B')
    except (ValueError,LessonError):pass
    else:raise AssertionError('练习B：bool不能是合法epoch')
    return {'exercise':'B','passed':True}

def check_exercise_c(fn):
    expected={'seq':2,'last_hash':'a'*64,'tick':2,
              'sessions':{'team__k':{'rid':'team__k','tenant':'team','key':'k',
                  'version':'stable','prompt_ids':[1,2,3],'max_new_tokens':3,
                  'generated':[4],'terminal':None}}}
    identity={'weights':{'stable':'f'*64}}
    wrapped={'version':VERSION,'model_identity':identity,'state':copy.deepcopy(expected)}
    require(fn(wrapped,digest(wrapped),identity,expected) is True,
            '练习C：完整一致的快照应放行')
    spoof=copy.deepcopy(wrapped)
    spoof['state']['sessions']['team__k']['generated']=[5]
    for args in ((spoof,digest(spoof),identity,expected),
                 (wrapped,'0'*64,identity,expected),
                 (wrapped,digest(wrapped),{'weights':{'stable':'0'*64}},expected)):
        try:fn(*args)
        except (ValueError,LessonError):pass
        else:raise AssertionError('练习C：快照篡改/模型身份错误没有被拒绝')
    return {'exercise':'C','passed':True}

In [22]:
def rpo_report_ex(last_ack_tick,recovered_tick,last_ack_seq,recovered_seq,budget_ticks):
    raise NotImplementedError('独立完成RPO差和ACK丢失计数')

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(check_exercise_a(rpo_report_ex))

### 练习 B：epoch 单主写入围栏

实现 `fence_ex(request_epoch, persisted_epoch, request_primary, persisted_primary)`。只有**两个epoch严格相等且主机ID相等**才返回 True，否则抛出 ValueError；不要把 `True` 这个bool当作合法epoch。

In [23]:
def fence_ex(request_epoch,persisted_epoch,request_primary,persisted_primary):
    raise NotImplementedError('独立实现epoch围栏')

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(check_exercise_b(fence_ex))

### 练习 C：快照 hash 与 WAL 前缀

实现 `snapshot_verify_ex(payload, sha256, expected_identity, wal_prefix_state)`：核对摘要、版本、模型身份和**完整逻辑状态**。注意：如果攻击者能改写payload并重新计算裸SHA，单独检查摘要仍不足以通过。

In [24]:
def snapshot_verify_ex(payload,sha256,expected_identity,wal_prefix_state):
    raise NotImplementedError('独立实现快照与可信WAL前缀核验')

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(check_exercise_c(snapshot_verify_ex))

## 15. 参考答案及独立检查

先完成自己的三个函数，再展开下面的参考实现。示例对输入契约进行显式校验；参考答案不会自动替换你的练习函数。

In [25]:
def answer_a(last_ack_tick,last_recovered_tick,last_ack_seq,recovered_seq,max_rpo_ticks):
    values=(last_ack_tick,last_recovered_tick,last_ack_seq,recovered_seq,max_rpo_ticks)
    require(all(type(v) is int and v>=0 for v in values),'计数/预算必须是非负整数')
    require(last_ack_tick>=last_recovered_tick and last_ack_seq>=recovered_seq,
            '恢复点不能比最后确认的状态还新')
    gap=last_ack_tick-last_recovered_tick
    return {'rpo_gap_ticks':gap,'lost_acked_events':last_ack_seq-recovered_seq,
            'within_budget':gap<=max_rpo_ticks}

def answer_b(request_epoch,persisted_epoch,requested_primary,persisted_primary):
    require(type(request_epoch) is int and request_epoch>0 and
            type(persisted_epoch) is int and persisted_epoch>0,'epoch非法')
    require(request_epoch==persisted_epoch and requested_primary==persisted_primary and
            requested_primary in {'A','B'},'旧主节点或错误epoch禁止写入')
    return True

def answer_c(payload,sha256,expected_identity,expected_wal_prefix_state):
    require(type(payload) is dict and sha256==digest(payload),'快照摘要不匹配')
    require(payload.get('version')==VERSION and
            payload.get('model_identity')==expected_identity,'快照model身份不同')
    require(payload.get('state')==expected_wal_prefix_state,
            '快照与可验证的WAL前缀不一致')
    return True

In [26]:
print('A:',check_exercise_a(answer_a))
print('B:',check_exercise_b(answer_b))
print('C:',check_exercise_c(answer_c))

A: {'exercise': 'A', 'passed': True}
B: {'exercise': 'B', 'passed': True}
C: {'exercise': 'C', 'passed': True}


## 16. 配套脚本命令

```bash
python lesson_81_disaster_recovery_snapshots_replica_consistency.py --checks-only
python lesson_81_disaster_recovery_snapshots_replica_consistency.py --math-only
python lesson_81_disaster_recovery_snapshots_replica_consistency.py
python lesson_81_disaster_recovery_snapshots_replica_consistency.py --reload-run lesson_81_outputs/实际运行目录
```

`--math-only` 不需要创建大规模演练目录；默认完整运行先执行固定检查，再运行10个真实KV场景。`

不要拿本课逻辑RTO tick作为毫秒或秒指标；若要在真实环境声明RTO/RPO，必须指定真实失效域、持久存储边界、客户端提交含义、时间采样与网络分区行为，并进行独立实测。

In [27]:
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    actions=parser.add_mutually_exclusive_group()
    actions.add_argument('--checks-only',action='store_true')
    actions.add_argument('--math-only',action='store_true')
    actions.add_argument('--reload-run',type=Path)
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_81_outputs'))
    parser.add_argument('--seed',type=int,default=8101)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2));return 0
    if args.math_only:
        print(json.dumps({'rpo_example':answer_a(15,12,15,12,3),
            'rto_formula':'2 + ceil(verified_records / record_rate) + ceil(rebuilt_token_count / token_rate)',
            'strict_sync':'ACK only after both local journals fsync',
            'async':'ACK primary first; backup may lag; explicit loss override required',
            'fencing':'single trusted local epoch pointer, NOT distributed consensus'},
            ensure_ascii=False,indent=2));return 0
    checks=run_checks()
    print('固定检查通过：',checks['passed'],'项；真实CPU KV + 双实例本地持久演练。',flush=True)
    if args.checks_only:return 0
    result=run_experiment(args.output_dir,args.seed)
    write_json(result/'checks.json',checks)
    return 0

## 17. 毕业验收与下一课

应该能够解释并亲手证明：

1. `sync` ACK 与仅主机 `fsync` 的区别；异步缺口为何必须报警而不能被自动洗掉。
2. 快照是逻辑token而非跨实例复用物理KV；为什么相同模型身份才能重算同一前缀。
3. epoch 只能在**可信单点控制者**下面形成教学围栏，不能代替真正的分布式共识。
4. “WAL已经写盘但ACK未送达”为什么需要同幂等键对账，不能自动重跑副作用。
5. 回滚/恢复后如何验收：数据完整性、逻辑RPO、逻辑RTO、KV全前向等价、页池释放。
6. 裸hash不等于数字签名，持久token IDs也不是已经加密的安全存储。

下一课建议：**第82课——混沌演练、灾备容量与恢复演练证据**，从已知故障扩展到多组合故障、资源恢复容量与独立故障注入报告。

## 参考资料

- [AWS Well-Architected：RTO与RPO定义](https://docs.aws.amazon.com/wellarchitected/latest/reliability-pillar/disaster-recovery-dr-objectives.html)
- [Google Cloud：测试数据丢失恢复，应同时核对数据完整性、RTO与RPO](https://docs.cloud.google.com/architecture/framework/reliability/perform-testing-for-recovery-from-data-loss)
- [Google Cloud：灾难恢复架构与要求](https://docs.cloud.google.com/architecture/disaster-recovery)
- [PyTorch：复现约束](https://docs.pytorch.org/docs/stable/notes/randomness.html)

文献用于概念定义；本课数值由本地真实代码生成，不是引用文献的生产指标。